<a href="https://colab.research.google.com/github/MarlzRana/machine-learning/blob/main/model_eval_segformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segformer Encoder + UNet Decode Evaluation

In this notebook we will be evaluating the model that utilizes a SegFormer encoder and UNet decoder

## Imports

External imports

In [ ]:
import torch

from torch.utils.data import DataLoader

from torchvision import transforms

Internal imports

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/mod_unet.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/dataset.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/loss.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/epochs.ipynb"

## Constants

In [ ]:
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

TEST_IMG_DIR = "./drive/MyDrive/CamVid/test"
TEST_MASK_DIR =  "./drive/MyDrive/CamVid/testannot"

CHECKPOINTS_DIR = "./drive/MyDrive/checkpoints/"

MODEL_SEGFORMER_UNET_CROP_CHECKPOINT = CHECKPOINTS_DIR + "model_segformer_unet_crop.pth"
MODEL_SEGFORMER_UNET_UPCONV_CHECKPOINT = CHECKPOINTS_DIR + "model_segformer_unet_upconv.pth"

## Testing Dataset

Define the image and mask transform

In [ ]:
img_transform = transforms.Compose(
    [
        transforms.Resize(size=(352, 352), antialias=True),
        lambda x: x.to(torch.float) / 255,
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

In [ ]:
mask_transform = transforms.Compose(
    [
        transforms.Resize(size=(352, 352), antialias=True),
    ]
)

Define the label map

In [ ]:
label_map = {
        "sky": 0,
        "building": 1,
        "pole": 2,
        "road": 3,
        "pavement": 4,
        "tree": 5,
        "signsymbol": 6,
        "fence": 7,
        "car": 8,
        "pedestrain": 9,
        "bicyclist": 10,
        "unlabelled": 11
}

Define the testing dataset

In [ ]:
test_ds = SegmentationDatasetSeperateMasks(
    img_dir=TEST_IMG_DIR,
    mask_dir=TEST_MASK_DIR,
    label_map=label_map,
    img_transform=img_transform,
    mask_transform=mask_transform,
    mask_type=torch.float,
)

## Test Dataloader

In [ ]:
test_dl = DataLoader(test_ds, batch_size=8)

## Loss Function

In [ ]:
loss_fn = DiceLoss()

## Prepare Pre-trained Models

## Segformer-UNet model that uses a cropping strategy to handle the channel mismatch

Load in the model

In [ ]:
model_segformer_unet_crop = torch.load(MODEL_SEGFORMER_UNET_CROP_CHECKPOINT).to(DEVICE)

Create an evaluation epoch


In [ ]:
valid_epoch_segformer_unet_crop = EvalEpoch(
    dataloader=test_dl,
    model=model_segformer_unet_crop,
    loss_fn=loss_fn,
    metrics=[],
    device=DEVICE
)

## Segformer-UNet model that uses a cropping strategy to handle the channel mismatch

Load in the model

In [ ]:
model_segformer_unet_upconv = torch.load(MODEL_SEGFORMER_UNET_UPCONV_CHECKPOINT)

Create an evaluation epoch

In [ ]:
valid_epoch_segformer_unet_upconv = EvalEpoch(
    dataloader=test_dl,
    model=model_segformer_unet_upconv,
    loss_fn=loss_fn,
    metrics=[],
    device=DEVICE
)

## Evaluate

## Segformer-UNet model that uses a cropping strategy to handle the channel mismatch

In [ ]:
valid_epoch_segformer_unet_crop.run()

100%|██████████| 233/233 [01:09<00:00,  3.37it/s, eval_loss=0.162]


0.1622264583905538

## Segformer-UNet model that uses a upconvolution strategy to handle the channel mismatch

In [ ]:
valid_epoch_segformer_unet_upconv.run()

100%|██████████| 233/233 [00:17<00:00, 13.48it/s, eval_loss=0.168]


0.16777130166689555